## импорты

In [1]:
import os
from tqdm.auto import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from sklearn.metrics import f1_score
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# tmux attach -t download_torch

import seaborn
seaborn.set_theme(palette='summer')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
num_workers = os.cpu_count() if device == "cpu" else 0
sample_rate = 16_000
device

'cuda'

In [2]:
train_df = pd.read_csv('../datasets/train.csv')
val_df = pd.read_csv('../datasets/val.csv')
test_df = pd.read_csv('../datasets/test.csv')

In [3]:
target_cols = ['happy', 'sad', 'anger', 'surprise', 'disgust', 'fear']

train_labels = (train_df[target_cols].values > 0).astype(int)
val_labels = (val_df[target_cols].values > 0).astype(int)
test_labels = (test_df[target_cols].values > 0).astype(int)

### создание класса FeatDataset

In [4]:
class FeatDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

    def __len__(self):
        return len(self.features)

### настройка воспроизводимости
- по мануалу: https://pytorch.org/docs/stable/notes/randomness.html

In [5]:
import random

def set_random_state(random_state = 0):
    torch.manual_seed(random_state)
    random.seed(random_state)
    np.random.seed(random_state)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(random_state)
        torch.cuda.manual_seed(random_state)

        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


random_state = 42
set_random_state(random_state)
torch.use_deterministic_algorithms(True)

In [6]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8
%env PYTHONHASHSEED=42

env: CUBLAS_WORKSPACE_CONFIG=:4096:8
env: PYTHONHASHSEED=42


## обучение моделей

### архитектура модели
- простая FFN.

In [7]:
class EmotionFFN(nn.Module):
    def __init__(self, input_size, hidden_size=256, output_size=6, dropout_prob=0.3):
        super(EmotionFFN, self).__init__()
        
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.BatchNorm1d(hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            
            nn.Linear(hidden_size, hidden_size // 2),
            nn.BatchNorm1d(hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(dropout_prob),
            
            nn.Linear(hidden_size // 2, output_size)
        )

    def forward(self, x):
        return self.network(x)

### train loop

In [15]:
def get_pos_weights(labels):
    num_positives = torch.sum(labels, dim=0)
    num_negatives = labels.shape[0] - num_positives
    
    pos_weight = num_negatives / (num_positives + 1e-5)
    return pos_weight

In [16]:
def train_model(model, train_loader, val_loader, model_name, num_epochs):
    model = model.to(device)
    
    weights = get_pos_weights(torch.tensor(train_labels, dtype=torch.float32))
    criterion = nn.BCEWithLogitsLoss(pos_weight=weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    
    best_f1 = 0
    history = {'train_loss': [], 'val_f1': []}

    log_filepath = f'../logs/{model_name}_training.log'
    with open(log_filepath, 'w') as f:
        f.write(f"Starting training for model: {model_name}\n")

    for epoch in range(num_epochs):
        # --- TRAIN ---
        model.train()
        train_losses = []
        for features, labels in tqdm(train_loader, desc=f"[{model_name}] Epoch {epoch}"):
            features, labels = features.to(device), labels.to(device).float()
            
            optimizer.zero_grad()
            logits = model(features)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # --- EVAL ---
        model.eval()
        val_preds, val_true = [], []
        with torch.no_grad():
            for v_feat, v_lab in val_loader:
                v_feat = v_feat.to(device)
                logits = model(v_feat)
                preds = (torch.sigmoid(logits) > 0.5).float()
                val_preds.append(preds.cpu())
                val_true.append(v_lab.cpu().float())

        y_true = torch.cat(val_true).numpy()
        y_pred = torch.cat(val_preds).numpy()
        
        f1_macro = f1_score(y_true, y_pred, average='macro', zero_division=0) * 100
        avg_loss = np.mean(train_losses)
        
        log_entry = f"Epoch {epoch} | Loss: {avg_loss:.4f} | F1 Macro: {f1_macro:.2f}%"
        with open(log_filepath, 'a') as f:
            f.write(log_entry + '\n')
        
        if f1_macro > best_f1:
            best_f1 = f1_macro
            torch.save(model.state_dict(), f'../models/best_model_{model_name}.pth')
            with open(log_filepath, 'a') as f:
                f.write(f"--- New Best F1: {best_f1:.2f}% (Saved Model) ---\n")
            
    print(f"Best F1: {best_f1:.2f}%")
    return best_f1

In [17]:
batch_size = 64
loader_args = {
    'batch_size': batch_size,
    'num_workers': num_workers
}

experiments = {
    'fasttext': (300, '../features/ft_'),
    'tfidf': (1000, '../features/tfidf_'),
    'mfcc': (40, '../features/scaled_mfcc_'),
    'mel': (256, '../features/scaled_mel_')
}

In [18]:
num_epochs = 50
overall_results = {}

for name, (input_dim, path_prefix) in experiments.items():
    print(f"Оценка модальности: {name.upper()}")
    
    train_feat = pd.read_csv(f'{path_prefix}train.csv').values
    val_feat = pd.read_csv(f'{path_prefix}val.csv').values
    
    train_ds = FeatDataset(train_feat, train_labels)
    val_ds = FeatDataset(val_feat, val_labels)
    
    train_loader = DataLoader(train_ds, shuffle=True, **loader_args)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_args)
    
    model = EmotionFFN(input_size=input_dim, output_size=6)
    
    best_score = train_model(model, train_loader, val_loader, name, num_epochs)
    overall_results[name] = best_score

print("\nСравнение моделей:")
for m_name, score in overall_results.items():
    print(f"{m_name}: Best F1 = {score:.2f}%")

Оценка модальности: FASTTEXT


[fasttext] Epoch 0:   0%|          | 0/255 [00:00<?, ?it/s]

[fasttext] Epoch 1:   0%|          | 0/255 [00:00<?, ?it/s]

[fasttext] Epoch 2:   0%|          | 0/255 [00:00<?, ?it/s]

[fasttext] Epoch 3:   0%|          | 0/255 [00:00<?, ?it/s]

...

[mel] Epoch 46:   0%|          | 0/255 [00:00<?, ?it/s]

[mel] Epoch 47:   0%|          | 0/255 [00:00<?, ?it/s]

[mel] Epoch 48:   0%|          | 0/255 [00:00<?, ?it/s]

[mel] Epoch 49:   0%|          | 0/255 [00:00<?, ?it/s]

Best F1: 37.09%

Сравнение моделей:
fasttext: Best F1 = 39.13%
tfidf: Best F1 = 37.47%
mfcc: Best F1 = 36.08%
mel: Best F1 = 37.09%
